# Filtering granules by file (band) patterns

Some datasets — like HLS — are **multi-file** granules: each granule contains several bands (e.g. `B02.tif`, `B03.tif`, `B04.tif`, `Fmask.tif`), a browse image, and metadata files. When you search, `earthaccess` returns the whole granule; this notebook shows how to pick just the files (bands) you want using the `AssetFilter` glob patterns, then download only those files.

## Setup

Login so `earthaccess` can build an authenticated session for indirect (HTTPS) access.

In [ ]:
import earthaccess

auth = earthaccess.login()
auth

## 1. Search for HLS granules

Each HLS granule is a set of files. `search_data()` returns granules; the files live inside each granule.

In [ ]:
granules = earthaccess.search_data(
    short_name="HLSL30",
    temporal=("2025-01-01", "2025-01-20"),
    bounding_box=(40.19, 6.24, 42.18, 7.24),
    count=2,
)
print(f"{len(granules)} granules")
granule = granules[0]

## 2. Look at the granule's files (assets)

`granule.assets()` returns the granule's files as `Asset` objects (with `href`, `roles`, `type`, …). `granule.data_assets()` returns only the `data`-role ones.

In [ ]:
for asset in granule.data_assets():
    print(asset.href.split("/")[-1], "|", asset.roles)

## 3. Select specific bands with `AssetFilter`

`earthaccess.store.AssetFilter` matches file names/URLs with **glob patterns** (the same syntax as shell `fnmatch`), plus role and size criteria.

To grab only the red/green/blue bands (`B02`, `B03`, `B04`):

In [ ]:
from earthaccess.store import AssetFilter

rgb = AssetFilter(include_patterns=["*B0[234].tif"])

selected = [a for a in granule.assets() if rgb.matches(a)]
for a in selected:
    print(a.href.split("/")[-1])

### Why `B0[234]`?

`AssetFilter` uses **glob** pattern matching, not regex:

| Pattern | Matches |
|---|---|
| `*B0[234].tif` | `B02`, `B03`, `B04` |
| `*B0[0-9].tif` | any `B0x` band |
| `*.tif` | every GeoTIFF |
| `*Fmask*` | the Fmask layer |
| `[Bb]02*` | case-insensitive `b02`/`B02` |

For regex-style matching you would need to match on the filename yourself; glob covers the common band-selection cases.

## 4. Exclude files (browse/thumbnail)

Combine include and exclude patterns. Here we keep every data file but drop the browse image:

In [ ]:
no_browse = AssetFilter(
    include_roles=["data"],
    exclude_patterns=["*browse*", "*thumb*"],
)

for a in granule.assets():
    if no_browse.matches(a):
        print(a.href.split("/")[-1])

## 5. Combine filters

`AssetFilter.combine()` unions patterns and tightens size bounds:

In [ ]:
bands = AssetFilter(include_patterns=["*B0[234].tif"])
data_only = AssetFilter(include_roles=["data"])
combined = bands.combine(data_only)

for a in granule.assets():
    if combined.matches(a):
        print(a.href.split("/")[-1])

## 6. Download only the selected bands

`earthaccess.download()` accepts raw URLs. Pass the selected band `href`s to fetch
only those files.

> **Note on credentials.** When you pass raw URLs, `earthaccess` has no granule
> metadata to infer S3 credentials from. For **in-region** `s3://` downloads you
> must supply `provider` (e.g. `"LPCLOUD"`) or `credentials_endpoint`; otherwise
> it cannot build the S3 filesystem. For **HTTPS** (out-of-region) downloads no
> S3 credentials are needed — the files are fetched through your authenticated
> EDL session.

When you pass `DataGranule` objects instead, `earthaccess` infers the provider
and credentials endpoint from the granule metadata automatically — but it
downloads **every** data file in the granule.


In [ ]:
selected_hrefs = [a.href for a in selected]
print("will download:")
for h in selected_hrefs:
    print(" ", h.split("/")[-1])

# Out-of-region / HTTPS: no provider needed
# earthaccess.download(selected_hrefs, "./data")

# In-region S3: pass the provider so credentials can be fetched
# earthaccess.download(selected_hrefs, "./data", provider="LPCLOUD")

## Summary

- `granule.assets()` / `granule.data_assets()` list a multi-file granule's files as `Asset` objects.
- `AssetFilter` selects files with **glob** patterns, roles, and size — `include_patterns` / `exclude_patterns`, `include_roles` / `exclude_roles`.
- Combine criteria with `AssetFilter.combine()`.
- Pass the selected `href`s to `earthaccess.download()` to fetch only those bands. Passing raw URLs means you must supply `provider` (or `credentials_endpoint`) for in-region S3; HTTPS downloads need no S3 credentials.

For server-side subsetting (when the dataset offers it), see [Search services](../../user/howto/search-services.md).
